# WRIT 20833 — Data Cleaning with Pandas

**When Coding Meets Culture: Developing Data-Driven Opinions**

Make an editable copy of this worksheet by going to **File > Save a copy in Drive**

Last class the data was *tidy*. Real found data almost never is. Scraped comments show up with
stray spaces, line breaks in the middle of text, the same category spelled five different ways, blank
cells, and duplicate rows. Before you can analyze anything, you have to **clean** it. Today you'll take
a messy version of the same comments table from last class and turn it back into something trustworthy —
and notice that every cleaning choice is also an *interpretive* choice.

# A Realistically Messy Dataset

Here's what those tidy YouTube comments might actually look like fresh out of a scraper. Same columns
as last time — `comment`, `stance`, `likes`, `replies` — but full of real-world junk.

In [ ]:
import pandas as pd
import numpy as np

data = {
    "comment": [
        "  The Ten Commandments belong in every classroom, period.  ",
        "This is a clear violation of church and state.\nKeep it out.",
        "Morals matter and kids today need them more than ever.",
        "Whose religion gets to decide? Not the government's job.",
        "Honestly I have no strong opinion either way.",
        "God and country, that's what built this nation.",
        "Public schools serve everyone, not just one faith.  ",
        "Put the Constitution in classrooms, not commandments.",
        "Finally some common sense values in our schools.",
        "Freedom of religion means freedom from it too.",
        "My kids should learn this at home, not at school.",
        "  The Ten Commandments belong in every classroom, period.  ",
    ],
    "stance": ["support", "Oppose", "SUPPORT", "against", "Neutral", "for",
               "oppose ", "OPPOSE", "support", "Against", "neutral", "support"],
    "likes": [240, 312, 88, 150, np.nan, 205, 176, 410, 64, 198, 33, 240],
    "replies": [15, 42, 6, 22, 1, 18, 19, 51, 4, 27, 2, 15],
}

messy_df = pd.DataFrame(data)
messy_df

# Step 1: Diagnose Before You Touch Anything

Good cleaning starts with *looking*. Three quick questions tell you most of what's wrong:
**What's missing? What's inconsistent? What's duplicated?**

In [ ]:
# what's missing?
print("Missing values per column:")
print(messy_df.isnull().sum())

In [ ]:
# what's inconsistent? .unique() shows every distinct value in a column
print("Stance labels as found:")
print(messy_df["stance"].unique())

In [ ]:
# what's duplicated? (a whole row repeated)
print("Duplicate rows:", messy_df.duplicated().sum())

So: one missing `likes` value, a `stance` column that means three things but is spelled ten
ways, and one repeated comment. Let's fix them one at a time on a **copy** (never destroy your raw
data — you may need to go back).

In [ ]:
clean_df = messy_df.copy()

# Step 2: Clean the Text Fields

Pandas applies a string method to a *whole column* at once through the `.str` accessor — it's the
column-wide version of the string methods you already know. Strip the whitespace, drop the stray
newline.

In [ ]:
clean_df["comment"] = clean_df["comment"].str.strip().str.replace("\n", " ", regex=False)
clean_df["comment"].tolist()

# Step 3: Standardize the Categories

`stance` is the worst offender. First flatten the casing and spacing, then **map** the synonyms
("for" → support, "against" → oppose) onto a clean set of three labels.

In [ ]:
# lowercase + trim so "OPPOSE", "Oppose", "oppose " all become "oppose"
clean_df["stance"] = clean_df["stance"].str.strip().str.lower()
print("After casing fix:", clean_df["stance"].unique())

In [ ]:
# map the synonyms onto canonical labels
clean_df["stance"] = clean_df["stance"].replace({"for": "support", "against": "oppose"})
print("Canonical labels:", clean_df["stance"].unique())

### Your turn

In [ ]:
# your turn — run .value_counts() on the cleaned stance column to see the three-way split
# clean_df["stance"].value_counts()

# Step 4: Handle the Missing Value

One comment has no `likes` count. You have choices, and **none are neutral**:
- **Drop** the row (`.dropna()`) — but you lose a real comment.
- **Fill** it (`.fillna()`) with an estimate — but you're inventing data.

Here we fill with the **median** (the middle value, less skewed by that 410-like outlier than the
mean). Whatever you choose, *say so* in your writeup — it's part of your method.

In [ ]:
median_likes = clean_df["likes"].median()
clean_df["likes"] = clean_df["likes"].fillna(median_likes)

print("Filled the missing likes with the median:", median_likes)
print("Any missing left?", clean_df["likes"].isnull().sum())

# Step 5: Drop the Duplicate

Now that the text is trimmed, the repeated comment is an *exact* duplicate. `.drop_duplicates()`
keeps the first and removes the rest.

In [ ]:
before = len(clean_df)
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
print(f"Rows: {before} -> {len(clean_df)}")
clean_df

# Putting It All Together: Clean Data Tells a Truer Story

The whole point of cleaning is that the analysis you couldn't trust before, you can trust now. With
the labels standardized and the duplicate gone, the counts finally mean something.

In [ ]:
print("Stance breakdown (clean):")
print(clean_df["stance"].value_counts())

print("\nAverage likes by stance:")
print(clean_df.groupby("stance")["likes"].mean().round(1))

Before cleaning, `.value_counts()` on `stance` would have shown *ten* fake categories and a
double-counted comment — a garbage picture. Same data, cleaned, and the crowd's split comes through.
You can also now reliably search the text — the `.str` accessor again, this time to *filter*:

In [ ]:
# which comments invoke "religion" directly?
religion = clean_df[clean_df["comment"].str.lower().str.contains("religion")]
religion[["comment", "stance"]]

In [ ]:
# your turn — filter to comments mentioning "schools" (or another word) and see who's on which side

# Sneak Preview: Where This Is Going

**This Friday's workshop** is you doing exactly this — to a dataset *you* collected. Diagnose it,
standardize it, drop the junk, until it's analysis-ready. Cleaning is unglamorous, but it's where
honest analysis begins: every choice you make here (what to fill, what to drop, how to label) quietly
shapes the conclusions you'll draw later.

And next week the payoff arrives: with a clean `comment` column, **VADER** will read each comment's
*sentiment* — turning "support / oppose / neutral" from something you labeled by hand into something
the computer scores. Clean data in, real analysis out.

In [ ]:
# a peek at next week: clean text is what sentiment scoring needs
clean_df[["comment", "stance"]].head()

# Playground

In [ ]:
# use this space to experiment!